In [1]:
import cv2
import numpy as np
import time
from scipy.spatial import distance as dist
from collections import OrderedDict

In [4]:
dataset_path = "../data/object_detection_classes_coco.txt"
model_path = "../data/frozen_inference_graph.pb"
weights_path = "../data/ssd_mobilenet_v2_coco_2018_03_29.pbtxt"

with open(dataset_path, "r") as f:
    class_names = f.read().split("\n")

model = cv2.dnn.readNetFromTensorflow(model_path, weights_path)

In [6]:
class CentroidTracker:
    def __init__(self, maxDisappeared=40):
        self.nextObjectID = 0
        self.objects = OrderedDict()
        self.disappeared = OrderedDict()
        self.maxDisappeared = maxDisappeared

    def register(self, centroid):
        self.objects[self.nextObjectID] = centroid
        self.disappeared[self.nextObjectID] = 0
        self.nextObjectID += 1

    def deregister(self, objectID):
        del self.objects[objectID]
        del self.disappeared[objectID]

    def update(self, rects):
        if len(rects) == 0:
            for objectID in list(self.disappeared.keys()):
                self.disappeared[objectID] += 1

                if self.disappeared[objectID] > self.maxDisappeared:
                    self.deregister(objectID)

            return self.objects

        inputCentroids = np.zeros((len(rects), 2), dtype="int")

        for (i, (startX, startY, endX, endY)) in enumerate(rects):
            cX = int((startX + endX) / 2.0)
            cY = int((startY + endY) / 2.0)
            inputCentroids[i] = (cX, cY)

        if len(self.objects) == 0:
            for i in range(len(inputCentroids)):
                self.register(inputCentroids[i])

        else:
            objectIDs = list(self.objects.keys())
            objectCentroids = list(self.objects.values())

            D = dist.cdist(np.array(objectCentroids),
                           inputCentroids)

            rows = D.min(axis=1).argsort()
            cols = D.argmin(axis=1)[rows]

            usedRows = set()
            usedCols = set()

            for (row, col) in zip(rows, cols):

                if row in usedRows or col in usedCols:
                    continue

                objectID = objectIDs[row]
                self.objects[objectID] = inputCentroids[col]
                self.disappeared[objectID] = 0

                usedRows.add(row)
                usedCols.add(col)

            unusedRows = set(range(0, D.shape[0])).difference(usedRows)
            unusedCols = set(range(0, D.shape[1])).difference(usedCols)

            if D.shape[0] >= D.shape[1]:
                for row in unusedRows:
                    objectID = objectIDs[row]
                    self.disappeared[objectID] += 1

                    if self.disappeared[objectID] > self.maxDisappeared:
                        self.deregister(objectID)
            else:
                for col in unusedCols:
                    self.register(inputCentroids[col])

        return self.objects

In [7]:
video = cv2.VideoCapture("https://172.16.2.222:8080/video")

ret, frame = video.read()
height, width, _ = frame.shape

tracker = CentroidTracker(maxDisappeared=100)

counted_people = {}
FOUR_HOURS = 4 * 60 * 60

total_people = 0

while True:
    ret, frame = video.read()
    if not ret:
        break

    current_time = time.time()

    for pid in list(counted_people.keys()):
        if current_time - counted_people[pid] > FOUR_HOURS:
            del counted_people[pid]

    blob = cv2.dnn.blobFromImage(
        frame,
        size=(300, 300),
        swapRB=True,
        crop=False
    )

    model.setInput(blob)
    output = model.forward()

    rects = []

    for detection in output[0, 0, :, :]:
        score = float(detection[2])

        if score > 0.6:
            class_id = int(detection[1])
            class_name = class_names[class_id - 1]

            if class_name == "person":
                left = int(detection[3] * width)
                top = int(detection[4] * height)
                right = int(detection[5] * width)
                bottom = int(detection[6] * height)

                rects.append((left, top, right, bottom))

                cv2.rectangle(
                    frame,
                    (left, top),
                    (right, bottom),
                    (125, 0, 200),
                    2
                )

    objects = tracker.update(rects)

    for objectID, centroid in objects.items():

        if objectID not in counted_people:
            counted_people[objectID] = current_time
            total_people += 1

        text = f"ID {objectID}"
        cv2.putText(
            frame,
            text,
            (centroid[0] - 10, centroid[1] - 10),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.5,
            (0, 255, 0),
            2
        )

        cv2.circle(
            frame,
            (centroid[0], centroid[1]),
            4,
            (0, 255, 0),
            -1
        )

    cv2.putText(
        frame,
        f"Total unique people: {total_people}",
        (20, 30),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.8,
        (0, 0, 255),
        2
    )

    cv2.imshow("Video", frame)

    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

video.release()
cv2.destroyAllWindows()